# Open Modelica controlled by Python
### 1. Overview
This script allows
- to load an OpenModelica Model
- to run a simulation for given conditions
- to store the results of the simulation together with the relevant simulation metadata in a JSON file. 


Dugr, 2025-11-19

#### Structure of the JSON file containing simulation results and metadata
```
{
    metadata:{
        para1: 13,
        para2: 'hallo',
        para3: 20
    }
    data:{
        [
            {
                'time':0,
                'coil1.p.v:0
            },
            {
                'time:0.01,
                'coil1.p.v:0
            }
        ]
    }
}
```


#### Comments on the code
**File littering**  
OpenModelica creates a large amount of files when it compiles a model. These files are stored in the working directory of OpenModelica and remain there after completion of the simulation when OM is called from OMPython. To avoid "littering" in the directory, the code relies on a temporary directory (Simulation_Bin) where all temporary files are stored. At the end of the simulation, the file containing the results is copied from the working_directory to the *result_directory*.
  
**Output format**  
Per default, OM stores the results in a matlab format file. This code specifies the outputFormat as .csv. 


In [1]:
from OMPython import OMCSessionZMQ
import pandas as pd
import matplotlib.pyplot as plt 
import json
import os
import shutil
from datetime import datetime, timezone
from pymongo import MongoClient

In [2]:
def list_to_dict(liste):
    """utility function to transform simulation option list to dict"""
    
    result = {}

    for item in liste:
        key, value = item.split("=", 1)
        key = key.strip()
        value = value.strip()

        # Remove surrounding quotes if present
        if (value.startswith("'") and value.endswith("'")) or \
        (value.startswith('"') and value.endswith('"')):
            value = value[1:-1]

        # Try converting to int or float
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass

        result[key] = value
    return result

zeitstamp = datetime.now(timezone.utc)                              # get the system time (in UTC)
timestamp = zeitstamp.strftime("%Y%m%d%H%M%SZ") 

# define the working directory
temp_working_dir = "C:/Users/gregor.dudle/AppData/Local/Temp/Simulation_Bin/"   # temporary directory where OpenModelica can store the temporary files

# base path of the Modelica Work
base_dir = "C:/Users/gregor.dudle/OneDrive - OST/aFE/2026_Kibble/"
results_dir = base_dir + "Results/"
print(f"results_dir: {results_dir}")

#################################################################################################
### define the components to be loaded                                                          
##
# Kibble Balance Dynamic Phase
kibble_balance_dir = base_dir+"Modelica_Work/DynamicPhase/"
kibble_balance_model = "DynamicPhaseV03"

# Coil
components_dir = base_dir + "Modelica_Work/Components/BIPM_Coil/"
coil_model = "BIPM_coil_2V1"

# Magnetic field
magnetic_dir = base_dir + "Modelica_Work/Components/BIPM_MagneticField/"
magnetic_field = "BIPM_magnetic_field_02"
magnetic_field_filename = magnetic_dir + magnetic_field + ".csv"

# Motion driver
driver_dir = base_dir + "Modelica_Work/Components/Motion_driver/"
x_driver_model = "triangular_position"


# create the list of files to be loaded
load_file_list = [
    kibble_balance_dir + kibble_balance_model+".mo",
    components_dir + coil_model +".mo",
    driver_dir + x_driver_model + ".mo"
]
##
### end of definition of the components
##################################################################################################


#################################################################################################
### define the parameters of the movement
##
amplitude = 0.004
offset = 0.0
period = 20


# define the result filenames
#   the OMPython will first write all outputs to workingDir+filename (=origin_file)
#   and store the simulation result together with the provided metadata to results_dir + "Simulation_" + timestamp + "_res.json"
#   if uncommented below the script will also copy the csv file to resultsDir
origin_file = temp_working_dir + kibble_balance_model + "_res.csv"
results_file_csv = results_dir + "Simulation_" + timestamp + "_res.csv"
results_file_json = results_dir + "Simulation_" +timestamp + "_res.json"

# create temp_working_dir 
try:
    os.mkdir(temp_working_dir)
except FileExistsError:
    print("Temporary directory already exists.")
    
print(f"Simulation data and metadata will be stored in 'Simulation_{timestamp}_res.json'")

# load model and simulate in Open Modelica
omc = OMCSessionZMQ()
print(omc.sendExpression("getVersion()"))
omc.sendExpression("cd(\""+temp_working_dir+"\")")

for filename in load_file_list:
    omc.sendExpression(f"loadFile(\"{filename}\")")
# the syntax is very subtle! do not remove quote or double back slashes as they are all required

simflags = (
    f'-override=magTableFileName={magnetic_field_filename},'        # magTableFileName is a parameter in the OpenModelica model DynamicPhaseV03
    f'x_pos_driver.amplitude={amplitude},'
    f'x_pos_driver.period={period}'
)
sim = omc.sendExpression(
    f'simulate({kibble_balance_model},'
    f'stopTime=100.0,'
    f'numberOfIntervals=5000,'
    f'simflags="{simflags}",'
    f'outputFormat="csv")'
)
#shutil.copy(origin_file,results_file_csv) # uncomment if the .csv is needed
print(sim)
print(
    omc.sendExpression(
        f'getParameterNames({kibble_balance_model})'
    )
)
# collect metadata of the simulation and store them together with the simulation data in the file 'Simulation_xxxxxxxxxxx_res.json'
metadata = {
    'id': timestamp,
    'date': datetime.today().strftime('%Y-%m-%d %H:%M:%S'),
    'user': os.getlogin(),
    'KibbleBalanceModel': kibble_balance_model,
    'CoilModel': coil_model,
    'MagneticField': magnetic_field,
    'Motion':{
        'Driver': x_driver_model,
        'Amplitude':amplitude,
        'Period':period,
        'Offset':offset,
    },
    
    'Simulation':  list_to_dict(str(sim["simulationOptions"]).split(", "))
    }

df = pd.read_csv(origin_file)

package ={
    'metadata':metadata,
    'data':df.to_dict(orient="records")
}
with open(results_file_json, "w", newline="") as f:
    w = f.write(json.dumps(package,indent=4))

# example how to use the plot function of OpenModelica
omc.sendExpression("plot(coil.p.v)")

# add simulation to DB
client = MongoClient("mongodb://localhost:27017/")
DB = client["Simu_Data_Test"]
collection = DB["Simu_Data"]
collection.insert_one(metadata)

results_dir: C:/Users/gregor.dudle/OneDrive - OST/aFE/2026_Kibble/Results/
Temporary directory already exists.
Simulation data and metadata will be stored in 'Simulation_20260911062226Z_res.json'


C:\Users\gregor.dudle\AppData\Local\Temp\ipykernel_2828\2142860568.py:97: DeprecationWarning: The class OMCSessionZMQ is depreciated and will be removed in future versions! Please use OMCSession* classes instead!
  omc = OMCSessionZMQ()


OpenModelica v1.25.1 (64-bit)
{'resultFile': '', 'simulationOptions': "startTime = 0.0, stopTime = 100.0, numberOfIntervals = 5000, tolerance = 1e-6, method = 'dassl', fileNamePrefix = 'DynamicPhaseV03', options = '', outputFormat = 'csv', variableFilter = '.*', cflags = '', simflags = '-override=magTableFileName=C:/Users/gregor.dudle/OneDrive - OST/aFE/2026_Kibble/Modelica_Work/Components/BIPM_MagneticField/BIPM_magnetic_field_02.csv,x_pos_driver.amplitude=0.004,x_pos_driver.period=20'", 'messages': 'Simulation execution failed for model: DynamicPhaseV03\nLOG_STDOUT        | warning | invalid command line option: -\nLOG_STDOUT        | info    | usage: C:\\Users\\gregor.dudle\\AppData\\Local\\Temp\\Simulation_Bin/DynamicPhaseV03.exe\n|                 | |       | | <-abortSlowSimulation>\n|                 | |       | |   aborts if the simulation chatters\n|                 | |       | | <-alarm=value> or <-alarm value>\n|                 | |       | |   aborts after the given number 

InsertOneResult(ObjectId('6aa39e51ae5c16e8f3d67a9b'), acknowledged=True)